In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from itertools import product
sns.set()

Building the SMA backtester class

In [22]:
class SMABACKTESTER():
    def __init__(self,symbol, sma_s,sma_l,start,end):
        self.symbol = symbol
        self.sma_s = sma_s
        self.sma_l = sma_l
        self.start = start
        self.end = end
        self.results = None
        self.get_data()
        self.prepare_data()

    def __repr__(self):
        return "SMABACKTESTER(symbol = {}, sma_s = {}, sma_l = {}, start = {}, end = {})".format(self.symbol,self.sma_s,self.sma_l,self.start,self.end)

    def get_data(self):
        raw = yf.download(tickers = self.symbol, start = self.start, end = self.end)["Adj Close"]
        df = raw.copy()
        df.columns.name = None
        df['Returns'] = np.log(df[self.symbol]/df[self.symbol].shift(1))
        self.data = df

    def prepare_data(self):
        data = self.data.copy()
        data['sma_s'] = data[self.symbol].rolling(self.sma_s).mean()
        data['sma_l'] = data[self.symbol].rolling(self.sma_l).mean()
        self.data = data

    def set_parameters(self,sma_s = None,sma_l = None):
        if sma_s is not None:
            self.sma_s = sma_s
            self.data['sma_s'] = self.data[self.symbol].rolling(self.sma_s).mean()

        if sma_l is not None:
            self.sma_l = sma_l
            self.data['sma_l'] = self.data[self.symbol].rolling(self.sma_l).mean()

    def test_strategy(self):
        data = self.data.copy().dropna()
        data['position'] = np.where(data["sma_s"] > data["sma_l"] , 1, -1)
        data['strategy'] = data["position"].shift(1)*data["Returns"]
        data.dropna(inplace=True)
        data['creturns']  = data["Returns"].cumsum().apply(np.exp)
        data['cstrategy'] = data["strategy"].cumsum().apply(np.exp)
        self.results = data
        perf = data["cstrategy"].iloc[-1]
        outperf = perf - data["creturns"].iloc[-1]
        return round(perf,4),round(outperf,4)

    def plot_results(self):
        if self.results is None:
            print("run test_strategy() first")
        else:
            title = "{}| sma_s = {} | sma_l = {}".format(self.symbol,self.sma_s,self.sma_l)
            self.results[['creturns','cstrategy']].plot(title = title, figsize = (12,8))


    def optimize_parameters(self, sma_s_range, sma_l_range):
        combinations = list(product(range(*sma_s_range),range(*sma_l_range)))

        results = []
        for comb in combinations:
            self.set_parameters(comb[0],comb[1])
            results.append(self.test_strategy()[0])

        best_perf = np.max(results)
        opt = combinations[np.argmax(results)]

        self.set_parameters(opt[0],opt[1])
        self.test_strategy()

        many_results = pd.DataFrame(data = combinations, columns = ['sma_s','sma_l'])
        many_results['performance'] = results

        return round(best_perf,4),opt